# Sprint 10 - Webinar 31 · Diseño de ETL en Python (con SQLite) · Data Analytics práctico

**Proyecto de la sesión:** construir un ETL sencillo (Extract → Transform → Load) que toma datos *crudos* de una tienda, los limpia y los carga a una base de datos SQLite.

> Nivel: **principiante**. Código simple, paso a paso, sin trucos.


## Fecha
> Completa aquí la fecha de la sesión.


## Objetivo de la sesión práctica (100 min)
Al finalizar esta clase, podrás:

1. Explicar qué es **ETL** (y cuándo usarlo).
2. Extraer datos desde archivos **CSV** y **JSON** usando `pandas`.
3. Aplicar transformaciones básicas (limpieza, tipos, validaciones, joins).
4. Cargar datos a una base **SQLite** y verificar que todo quedó bien.
5. Armar un **pipeline** simple (funciones) para repetir el proceso.


## Agenda sugerida (100 minutos)
1. Contexto + checklist de entregables (5 min)
2. **Actividad 0 (breakout rooms):** diseñar el ETL en papel (10 min)
3. Ejercicio 1: crear datos crudos + estructura del proyecto (15 min)
4. Ejercicio 2: **Extract** (CSV + JSON) + exploración rápida (15 min)
5. Ejercicio 3: **Transform** (limpieza + modelo final) (20 min)
6. Ejercicio 4: **Load** a SQLite + consultas de verificación (20 min)
7. Ejercicio 5: mini-orquestación (pipeline) + re-ejecución (10 min)
8. Takeaways + cierre (5 min)


## Contexto
Imagina que trabajas como Data Analyst en una tienda online. Cada día recibes datos desde diferentes fuentes:

- Un archivo `customers_raw.csv` exportado desde un CRM.
- Un archivo `orders_raw.csv` exportado desde el sistema de ventas.
- Un archivo `products_raw.json` con el catálogo de productos.

El problema: **los datos vienen sucios** (espacios, mayúsculas, tipos incorrectos, fechas raras, valores faltantes).

Tu objetivo hoy es diseñar un ETL para:

1) **Extract**: leer los archivos crudos tal como llegan.
2) **Transform**: limpiar, estandarizar y construir tablas listas para análisis.
3) **Load**: cargar el resultado a una base SQLite para consultas.

### ¿ETL vs ELT?
- **ETL**: transformas *antes* de cargar (común en pipelines tradicionales y cuando el destino es simple).
- **ELT**: cargas primero y transformas después (muy común en data warehouses modernos).

Hoy haremos **ETL** porque:
- SQLite es liviano (ideal para practicar),
- queremos entender bien qué significa “transformar”,
- y necesitamos un resultado listo para análisis.


## Checklist de entregables (al final de la clase)
- [ ] Carpeta `data/raw/` con archivos crudos (CSV/JSON).
- [ ] Carpeta `data/processed/` con un dataset limpio (CSV).
- [ ] Base `data/db/tienda_etl.db` con tablas cargadas.
- [ ] 2–3 consultas SQL de verificación (conteos, joins).
- [ ] Función `run_etl()` que ejecute el pipeline completo.


---

# Actividad 0 · Calentamiento (10 min)

Antes de tocar código, diseñemos el ETL en equipo.

### Instrucciones (breakout rooms, 3–4 personas)
1) Lean el objetivo: **“queremos analizar ventas por cliente y por producto”**.
2) Respondan:
   - ¿Qué tablas finales (destino) necesitamos? (pista: *dimensiones* y *hechos*)
   - ¿Qué columnas son claves? (pista: `customer_id`, `order_id`, `product_id`)
   - ¿Qué problemas de calidad podrían venir en los datos crudos?
   - ¿Qué transformaciones mínimas harías?
3) Escriban una propuesta corta (5–8 líneas).

### Pista
Un modelo típico para análisis de ventas:
- `dim_customers` (una fila por cliente)
- `dim_products` (una fila por producto)
- `fact_orders` (una fila por orden o por ítem)

### Solución ejemplo (respuesta esperada)
- Tablas destino: `dim_customers`, `dim_products`, `fact_orders`.
- Claves: `customer_id`, `product_id`, `order_id`.
- Problemas: duplicados, emails con mayúsculas/espacios, fechas como texto, precios con símbolos, nulos.
- Transformaciones: renombrar columnas, `strip()` espacios, convertir fechas a `datetime`, convertir precios a número, validar rangos y eliminar/ajustar registros inválidos.


In [ ]:
# ============================================================
# Imports y configuración (ejecuta esta celda primero)
# ============================================================

import json
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


---

# Ejercicio 1 · Crear datos crudos + estructura del proyecto (15 min)

**Meta:** crear una carpeta de proyecto reproducible y generar archivos crudos con *errores típicos*.

### ¿Por qué separamos `raw/` y `processed/`?
- `raw/`: datos originales **sin modificar** (si algo sale mal, puedes volver a empezar)
- `processed/`: datos listos para análisis (resultado del ETL)
- `db/`: base de datos con tablas cargadas

### Estructura que construiremos
```
data/
  raw/
  processed/
  db/
```

### Tu turno
1) Crea las carpetas.
2) Genera 3 archivos:
   - `customers_raw.csv`
   - `orders_raw.csv`
   - `products_raw.json`

Los datos pueden ser sintéticos (inventados), pero deben incluir algunos problemas: espacios, mayúsculas, valores faltantes, fechas como texto, etc.

### Pista
- Usa `Path(...).mkdir(parents=True, exist_ok=True)`.
- Puedes crear `DataFrame`s y luego `to_csv(...)`.
- Para JSON, puedes guardar una lista de diccionarios.


In [ ]:
# Tu turno: crea carpetas y genera los archivos crudos (raw)
# TIP: si algo falla, vuelve a ejecutar la celda: debe ser segura (idempotente).

# 1) Crea carpetas: data/raw, data/processed, data/db
# 2) Crea dataframes para customers y orders
# 3) Guarda customers_raw.csv y orders_raw.csv
# 4) Crea products_raw.json (lista de productos)

# Escribe tu solución aquí 👇


In [ ]:
# =========================
# Solución · Ejercicio 1
# =========================

# 1) Carpetas
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
DB_DIR = Path("data/db")

for p in [RAW_DIR, PROCESSED_DIR, DB_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# 2) Customers (con problemas intencionales)
n_customers = 30
customers = pd.DataFrame({
    "Customer ID": [f"C{1000+i}" for i in range(n_customers)],
    "Name": np.random.choice([" Ana ", "JUAN", "carlos", "María  ", "   Pedro"], size=n_customers),
    "Email": np.random.choice(["ana@example.com", "JUAN@EXAMPLE.COM ", " carlos@example.com", None, "maria@example.com"], size=n_customers),
    "City": np.random.choice(["Bogotá", "Medellín", "Cali", "Barranquilla", "  bogotá "], size=n_customers),
})

# Duplicado intencional
customers = pd.concat([customers, customers.iloc[[0]]], ignore_index=True)

customers_path = RAW_DIR / "customers_raw.csv"
customers.to_csv(customers_path, index=False)

# 3) Products (JSON)
products = [
    {"product_id": "P1", "product_name": "Camiseta", "category": "Ropa", "unit_price": "39.900"},
    {"product_id": "P2", "product_name": "Gorra", "category": "Ropa", "unit_price": "24.500"},
    {"product_id": "P3", "product_name": "Termo", "category": "Accesorios", "unit_price": "49.900"},
    {"product_id": "P4", "product_name": "Medias", "category": "Ropa", "unit_price": None},  # nulo intencional
]

products_path = RAW_DIR / "products_raw.json"
with open(products_path, "w", encoding="utf-8") as f:
    json.dump(products, f, ensure_ascii=False, indent=2)

# 4) Orders (con problemas intencionales)
n_orders = 80
orders = pd.DataFrame({
    "order_id": [f"O{2000+i}" for i in range(n_orders)],
    "customer_id": np.random.choice(customers["Customer ID"].values, size=n_orders),
    "product_id": np.random.choice([p["product_id"] for p in products], size=n_orders),
    "order_date": np.random.choice(["2026-02-01", "01/02/2026", "2026/02/03", "2026-02-04 "], size=n_orders),
    "quantity": np.random.choice([1, 2, 3, -1, None], size=n_orders),  # negativos/nulos intencional
    "unit_price": np.random.choice(["39.900", "24.500", "49.900", "$39,900", None], size=n_orders),  # formatos variados
})

orders_path = RAW_DIR / "orders_raw.csv"
orders.to_csv(orders_path, index=False)

customers_path, orders_path, products_path


---

# Ejercicio 2 · Extract (CSV + JSON) + exploración rápida (15 min)

**Meta:** leer datos crudos y entender su forma (filas/columnas, tipos, nulos).

### Fundamento teórico (muy corto)
- En **Extract**, tu prioridad es *no perder información*. Si el dato viene como texto raro, lo lees como texto y lo transformas después.
- Las primeras preguntas siempre son:
  1) ¿Cuántas filas/columnas tengo?
  2) ¿Qué tipos detectó Python?
  3) ¿Qué tan sucios están? (nulos, duplicados, valores fuera de rango)

### Tu turno
1) Carga `customers_raw.csv` y `orders_raw.csv` con pandas.
2) Carga `products_raw.json`.
3) Muestra: `head()`, `shape`, `info()` y conteo de nulos por columna.

### Pista
- CSV: `pd.read_csv(ruta)`
- JSON: `json.load(open(...))` y luego `pd.DataFrame(lista)`


In [ ]:
# Tu turno: carga y explora los datos crudos

# 1) Carga customers_raw.csv y orders_raw.csv
# 2) Carga products_raw.json
# 3) Explora: head, shape, info, nulos por columna

# Escribe tu solución aquí 👇


In [ ]:
# =========================
# Solución · Ejercicio 2
# =========================

customers_raw = pd.read_csv(RAW_DIR / "customers_raw.csv")
orders_raw = pd.read_csv(RAW_DIR / "orders_raw.csv")

with open(RAW_DIR / "products_raw.json", "r", encoding="utf-8") as f:
    products_list = json.load(f)

products_raw = pd.DataFrame(products_list)

print("customers_raw:", customers_raw.shape)
display(customers_raw.head())

print("\norders_raw:", orders_raw.shape)
display(orders_raw.head())

print("\nproducts_raw:", products_raw.shape)
display(products_raw.head())

print("\n--- INFO customers ---")
customers_raw.info()

print("\nNulos customers:")
display(customers_raw.isna().sum().sort_values(ascending=False))

print("\n--- INFO orders ---")
orders_raw.info()

print("\nNulos orders:")
display(orders_raw.isna().sum().sort_values(ascending=False))


---

# Ejercicio 3 · Transform (limpieza + modelo final) (20 min)

**Meta:** convertir datos crudos en tablas limpias para análisis.

### Fundamento teórico
Transform es donde definimos **reglas de negocio** y **reglas de calidad**. Ejemplos:
- Estandarizar nombres de columnas (snake_case)
- Limpiar strings (`strip`, `lower`, `title`)
- Convertir tipos (`to_datetime`, `to_numeric`)
- Eliminar/ajustar registros inválidos (ej. `quantity <= 0`)
- Crear columnas derivadas (ej. `total = quantity * unit_price`)

### Tu turno
1) Limpia `customers_raw` para crear `dim_customers`.
2) Limpia `products_raw` para crear `dim_products`.
3) Limpia `orders_raw` para crear `fact_orders`.
4) Guarda un CSV limpio en `data/processed/orders_clean.csv`.

### Pista
- Para renombrar columnas: `df.rename(columns={...})`
- Para limpiar texto: `df['col'].astype(str).str.strip().str.lower()`
- Para números: `pd.to_numeric(..., errors='coerce')`
- Para fechas: `pd.to_datetime(..., errors='coerce', dayfirst=True)`


In [ ]:
# Tu turno: crea dim_customers, dim_products y fact_orders

# Reglas mínimas sugeridas:
# - Columnas en snake_case
# - Emails en minúscula y sin espacios
# - City con strip y formato tipo título (Title Case)
# - unit_price numérico (float)
# - order_date a datetime
# - quantity a entero; elimina quantity <= 0 o nulos
# - total = quantity * unit_price
# - Elimina duplicados en customers por customer_id

# Escribe tu solución aquí 👇


In [ ]:
# =========================
# Solución · Ejercicio 3
# =========================

# --- dim_customers ---
dim_customers = customers_raw.rename(columns={
    "Customer ID": "customer_id",
    "Name": "name",
    "Email": "email",
    "City": "city",
}).copy()

dim_customers["customer_id"] = dim_customers["customer_id"].astype(str).str.strip()

dim_customers["name"] = (
    dim_customers["name"]
    .astype(str)
    .str.strip()
    .str.title()
)

dim_customers["email"] = (
    dim_customers["email"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"none": np.nan, "nan": np.nan})
)

dim_customers["city"] = (
    dim_customers["city"]
    .astype(str)
    .str.strip()
    .str.title()
)

# Eliminar duplicados por customer_id (nos quedamos con la primera ocurrencia)
dim_customers = dim_customers.drop_duplicates(subset=["customer_id"], keep="first")

# --- dim_products ---
dim_products = products_raw.copy()
dim_products["product_id"] = dim_products["product_id"].astype(str).str.strip()

dim_products["product_name"] = dim_products["product_name"].astype(str).str.strip().str.title()
dim_products["category"] = dim_products["category"].astype(str).str.strip().str.title()

# unit_price viene como texto, a veces con puntos o nulos
dim_products["unit_price"] = (
    dim_products["unit_price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.replace(".", "", regex=False)   # '39.900' -> '39900'
)
dim_products["unit_price"] = pd.to_numeric(dim_products["unit_price"], errors="coerce")

# --- fact_orders ---
fact_orders = orders_raw.copy()

# Normaliza nombres de columnas
fact_orders = fact_orders.rename(columns={
    "order_id": "order_id",
    "customer_id": "customer_id",
    "product_id": "product_id",
    "order_date": "order_date",
    "quantity": "quantity",
    "unit_price": "unit_price",
})

# Limpieza básica de ids
for col in ["order_id", "customer_id", "product_id"]:
    fact_orders[col] = fact_orders[col].astype(str).str.strip()

# order_date a datetime: soporta varios formatos, usa dayfirst por si viene dd/mm/yyyy
fact_orders["order_date"] = fact_orders["order_date"].astype(str).str.strip()
fact_orders["order_date"] = pd.to_datetime(fact_orders["order_date"], errors="coerce", dayfirst=True)

# quantity a número
fact_orders["quantity"] = pd.to_numeric(fact_orders["quantity"], errors="coerce")
fact_orders["quantity"] = fact_orders["quantity"].astype("Int64")  # permite nulos

# unit_price: puede venir con símbolos
fact_orders["unit_price"] = (
    fact_orders["unit_price"]
    .astype(str)
    .str.strip()
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.replace(".", "", regex=False)
)
fact_orders["unit_price"] = pd.to_numeric(fact_orders["unit_price"], errors="coerce")

# Reglas de calidad mínimas
fact_orders = fact_orders.dropna(subset=["order_id", "customer_id", "product_id", "order_date", "quantity", "unit_price"])
fact_orders = fact_orders[fact_orders["quantity"] > 0]
fact_orders = fact_orders[fact_orders["unit_price"] > 0]

# Columna derivada
fact_orders["total"] = fact_orders["quantity"] * fact_orders["unit_price"]

# Guardar CSV procesado
processed_path = PROCESSED_DIR / "orders_clean.csv"
fact_orders.to_csv(processed_path, index=False)

display(dim_customers.head())
display(dim_products.head())
display(fact_orders.head())

processed_path


---

# Ejercicio 4 · Load a SQLite + consultas de verificación (20 min)

**Meta:** cargar `dim_customers`, `dim_products` y `fact_orders` a SQLite.

### Fundamento teórico
Una base SQL te permite:
- almacenar datos con estructura y reglas (tipos, llaves)
- hacer consultas reproducibles (`GROUP BY`, `JOIN`, etc.)
- compartir resultados con otras herramientas

SQLite es ideal para practicar porque:
- no requiere instalar un servidor
- es un archivo `.db`

### Tu turno
1) Crea una base `data/db/tienda_etl.db`.
2) Carga las 3 tablas.
3) Haz 3 consultas de verificación:
   - Conteo de filas por tabla
   - Total de ventas (sumatoria de `total`)
   - Top 5 productos por ventas

### Pista
- Conexión: `sqlite3.connect('ruta.db')`
- Cargar con pandas: `df.to_sql('tabla', conn, if_exists='replace', index=False)`
- Consultar: `pd.read_sql_query('SELECT ...', conn)`


In [ ]:
# Tu turno: crea la base y carga las tablas

# 1) Conecta/crea la base de datos
# 2) Carga dim_customers, dim_products, fact_orders
# 3) Corre consultas de verificación

# Escribe tu solución aquí 👇


In [ ]:
# =========================
# Solución · Ejercicio 4
# =========================

db_path = DB_DIR / "tienda_etl.db"
conn = sqlite3.connect(db_path)

# Carga tablas (full refresh)
dim_customers.to_sql("dim_customers", conn, if_exists="replace", index=False)
dim_products.to_sql("dim_products", conn, if_exists="replace", index=False)
fact_orders.to_sql("fact_orders", conn, if_exists="replace", index=False)

# 1) Conteo de filas por tabla
q_counts = '''
SELECT 'dim_customers' AS table_name, COUNT(*) AS n_rows FROM dim_customers
UNION ALL
SELECT 'dim_products'  AS table_name, COUNT(*) AS n_rows FROM dim_products
UNION ALL
SELECT 'fact_orders'   AS table_name, COUNT(*) AS n_rows FROM fact_orders;
'''
display(pd.read_sql_query(q_counts, conn))

# 2) Total de ventas
q_total_sales = "SELECT SUM(total) AS total_sales FROM fact_orders;"
display(pd.read_sql_query(q_total_sales, conn))

# 3) Top 5 productos por ventas
q_top_products = '''
SELECT
  p.product_id,
  p.product_name,
  SUM(o.total) AS sales
FROM fact_orders o
JOIN dim_products p USING(product_id)
GROUP BY p.product_id, p.product_name
ORDER BY sales DESC
LIMIT 5;
'''
display(pd.read_sql_query(q_top_products, conn))

conn.close()
db_path


---

# Ejercicio 5 · Mini-orquestación (pipeline) + re-ejecución (10 min)

**Meta:** organizar el ETL en funciones para que sea repetible.

### Fundamento teórico
Un buen ETL (aunque sea pequeño) debe ser:
- **repetible** (puedes correrlo 10 veces y siempre funciona)
- **legible** (otra persona entiende qué hace)
- **validable** (tiene chequeos mínimos)

### Tu turno
1) Crea 3 funciones: `extract()`, `transform()`, `load()`.
2) Crea `run_etl()` que ejecute todo.
3) Ejecuta `run_etl()` y verifica que crea el `.db`.

### Pista
- `extract()` debería devolver los dataframes crudos.
- `transform()` debería recibir crudos y devolver limpios.
- `load()` debería recibir limpios y escribir en SQLite.


In [ ]:
# Tu turno: organiza el ETL en funciones y ejecuta run_etl()

# Escribe tu solución aquí 👇


In [ ]:
# =========================
# Solución · Ejercicio 5
# =========================

def extract(raw_dir: Path):
    """Lee archivos crudos y devuelve (customers_raw, orders_raw, products_raw)."""
    customers_raw = pd.read_csv(raw_dir / "customers_raw.csv")
    orders_raw = pd.read_csv(raw_dir / "orders_raw.csv")
    with open(raw_dir / "products_raw.json", "r", encoding="utf-8") as f:
        products_raw = pd.DataFrame(json.load(f))
    return customers_raw, orders_raw, products_raw


def transform(customers_raw: pd.DataFrame, orders_raw: pd.DataFrame, products_raw: pd.DataFrame):
    """Limpia y construye dim_customers, dim_products, fact_orders."""

    # dim_customers
    dim_customers = customers_raw.rename(columns={
        "Customer ID": "customer_id",
        "Name": "name",
        "Email": "email",
        "City": "city",
    }).copy()

    dim_customers["customer_id"] = dim_customers["customer_id"].astype(str).str.strip()
    dim_customers["name"] = dim_customers["name"].astype(str).str.strip().str.title()
    dim_customers["email"] = (
        dim_customers["email"].astype(str).str.strip().str.lower()
        .replace({"none": np.nan, "nan": np.nan})
    )
    dim_customers["city"] = dim_customers["city"].astype(str).str.strip().str.title()
    dim_customers = dim_customers.drop_duplicates(subset=["customer_id"], keep="first")

    # dim_products
    dim_products = products_raw.copy()
    dim_products["product_id"] = dim_products["product_id"].astype(str).str.strip()
    dim_products["product_name"] = dim_products["product_name"].astype(str).str.strip().str.title()
    dim_products["category"] = dim_products["category"].astype(str).str.strip().str.title()
    dim_products["unit_price"] = (
        dim_products["unit_price"].astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace(".", "", regex=False)
    )
    dim_products["unit_price"] = pd.to_numeric(dim_products["unit_price"], errors="coerce")

    # fact_orders
    fact_orders = orders_raw.copy()
    for col in ["order_id", "customer_id", "product_id"]:
        fact_orders[col] = fact_orders[col].astype(str).str.strip()

    fact_orders["order_date"] = pd.to_datetime(fact_orders["order_date"].astype(str).str.strip(), errors="coerce", dayfirst=True)
    fact_orders["quantity"] = pd.to_numeric(fact_orders["quantity"], errors="coerce")
    fact_orders["unit_price"] = (
        fact_orders["unit_price"].astype(str).str.strip()
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace(".", "", regex=False)
    )
    fact_orders["unit_price"] = pd.to_numeric(fact_orders["unit_price"], errors="coerce")

    fact_orders = fact_orders.dropna(subset=["order_id", "customer_id", "product_id", "order_date", "quantity", "unit_price"])
    fact_orders = fact_orders[fact_orders["quantity"] > 0]
    fact_orders = fact_orders[fact_orders["unit_price"] > 0]
    fact_orders["quantity"] = fact_orders["quantity"].astype(int)
    fact_orders["total"] = fact_orders["quantity"] * fact_orders["unit_price"]

    return dim_customers, dim_products, fact_orders


def load(dim_customers: pd.DataFrame, dim_products: pd.DataFrame, fact_orders: pd.DataFrame, db_path: Path):
    """Carga tablas a SQLite (full refresh)."""
    conn = sqlite3.connect(db_path)
    dim_customers.to_sql("dim_customers", conn, if_exists="replace", index=False)
    dim_products.to_sql("dim_products", conn, if_exists="replace", index=False)
    fact_orders.to_sql("fact_orders", conn, if_exists="replace", index=False)
    conn.close()


def run_etl():
    customers_raw, orders_raw, products_raw = extract(RAW_DIR)
    dim_customers, dim_products, fact_orders = transform(customers_raw, orders_raw, products_raw)

    # Guarda un artefacto procesado (opcional, pero útil para debug)
    fact_orders.to_csv(PROCESSED_DIR / "orders_clean.csv", index=False)

    load(dim_customers, dim_products, fact_orders, DB_DIR / "tienda_etl.db")
    return DB_DIR / "tienda_etl.db"

run_etl()


## Takeaways de la sesión práctica
- ETL = **Extract → Transform → Load**: leer sin perder info, limpiar con reglas claras, cargar en un destino consultable.
- Separar `raw/` y `processed/` hace el proceso **repetible** y fácil de depurar.
- Las transformaciones mínimas para principiantes suelen ser: limpiar texto, convertir tipos, eliminar registros inválidos.
- SQLite es un excelente primer destino para practicar modelado y consultas.


## Cierre y próximos pasos
1) Tarea sugerida: agrega una validación extra (ej. que `product_id` de orders exista en products).
2) Bonus: crea una tabla `ventas_por_dia` con SQL (`GROUP BY date(order_date)`).
3) Próxima sesión: **automatización** del ETL (logging, manejo de errores, ejecución programada).
